# Setup: Install Dependencies

Run this notebook first to install all required packages.


In [1]:
# Install required packages
%pip install datasets matplotlib plotly pandas numpy torch transformers peft accelerate

In [2]:
import os
import sys

# ⚠️ CRITICAL: Set HF_HOME BEFORE importing transformers
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted successfully")
    # Updated path to CSI_Project with models subdirectory
    models_dir = "/content/drive/MyDrive/CSI_Project/models"
    datasets_dir = "/content/drive/MyDrive/CSI_Project/datasets"
except ImportError:
    # Local environment fallback
    models_dir = os.path.expanduser("~/.cache/huggingface/models")
    datasets_dir = os.path.expanduser("~/.cache/huggingface/datasets")
    print("⚠ Not running in Colab; using local cache directory")

# Ensure directories exist
os.makedirs(models_dir, exist_ok=True)
os.makedirs(datasets_dir, exist_ok=True)

# Set environment variables BEFORE importing transformers
os.environ['HF_HOME'] = models_dir
os.environ['TRANSFORMERS_CACHE'] = models_dir
os.environ['HF_DATASETS_CACHE'] = datasets_dir

print(f"Models directory: {models_dir}")
print(f"Datasets directory: {datasets_dir}\n")

# NOW import transformers (after env vars are set)
from transformers import AutoTokenizer, AutoModel

# Models to download for dual-encoder architecture
models = [
    "microsoft/graphcodebert-base",  # Primary: Code understanding via GNN
    "microsoft/codebert-base",  # Secondary: Context-aware code encoder
]

for model_name in models:
    print(f"{'='*60}")
    print(f"Downloading {model_name}...")
    print(f"{'='*60}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

        print(f"✓ Successfully downloaded {model_name}")
        print(f"  Model class: {model.__class__.__name__}")
        print(f"  Tokenizer class: {tokenizer.__class__.__name__}")
        print(
            f"  Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters"
        )
        print()

    except Exception as e:
        print(f"✗ Error downloading {model_name}: {e}")
        raise

print("\n" + "=" * 60)
print("✓ All models downloaded successfully!")
print(f"  Models location: {models_dir}")
print(f"  Datasets location: {datasets_dir}")
print("=" * 60)

Mounted at /content/drive
✓ Google Drive mounted successfully
Models directory: /content/drive/MyDrive/CSI_Project/models
Datasets directory: /content/drive/MyDrive/CSI_Project/datasets



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Successfully downloaded microsoft/codebert-base
  Model class: RobertaModel
  Tokenizer class: RobertaTokenizer
  Model size: 124.6M parameters


✓ All models downloaded successfully!
  Models location: /content/drive/MyDrive/CSI_Project/models
  Datasets location: /content/drive/MyDrive/CSI_Project/datasets


## Download HuggingFace Models

✓ All packages and 2 HuggingFace models downloaded:
- **GraphCodeBERT** (primary encoder)
- **CodeBERT** (secondary encoder for dual-encoder fusion)
- **PEFT + Accelerate** (LoRA fine-tuning support)

Now run `01_dataset_pipeline_local.ipynb`

In [4]:
import os
import shutil
import subprocess

print("=" * 60)
print("MOVING MODELS FROM COLAB CACHE TO GOOGLE DRIVE")
print("=" * 60)

# Source (Colab default cache)
source_cache = "/root/.cache/huggingface/hub"

# Destination (Google Drive - CSI_Project/models)
dest_cache = "/content/drive/MyDrive/CSI_Project/models"

if os.path.exists(source_cache):
    print(f"\n📦 Found models in Colab cache: {source_cache}")
    
    # Get source size
    result = subprocess.run(['du', '-sh', source_cache], capture_output=True, text=True)
    print(f"Size to copy: {result.stdout.strip()}")
    
    # Create destination structure
    os.makedirs(dest_cache, exist_ok=True)
    
    # Copy models
    print(f"\n⏳ Copying to Google Drive (this may take 2-5 minutes)...")
    try:
        # Copy all contents
        for item in os.listdir(source_cache):
            src = os.path.join(source_cache, item)
            dst = os.path.join(dest_cache, item)
            
            if os.path.isdir(src):
                if os.path.exists(dst):
                    print(f"  ⚠️  Skipping {item} (already exists)")
                else:
                    print(f"  📁 Copying {item}...")
                    shutil.copytree(src, dst)
            else:
                if not os.path.exists(dst):
                    print(f"  📄 Copying {item}...")
                    shutil.copy2(src, dst)
        
        print("\n✓ Copy complete!")
        
        # Verify
        result = subprocess.run(['du', '-sh', dest_cache], capture_output=True, text=True)
        print(f"\n📊 Google Drive models size: {result.stdout.strip()}")
        
        # List what's there
        print(f"\nContents of CSI_Project/models/:")
        for item in os.listdir(dest_cache):
            item_path = os.path.join(dest_cache, item)
            if os.path.isdir(item_path):
                result = subprocess.run(['du', '-sh', item_path], capture_output=True, text=True)
                size = result.stdout.strip().split('\t')[0]
                print(f"  📁 {item}/ ({size})")
        
        print("\n✓ Models are now on your Google Drive at CSI_Project/models/!")
        
    except Exception as e:
        print(f"✗ Error copying: {e}")
        raise
else:
    print(f"✗ No models found in Colab cache: {source_cache}")
    print("  Please run Cell 3 (Download HuggingFace Models) first")

MOVING MODELS FROM COLAB CACHE TO GOOGLE DRIVE
✗ No models found in Colab cache: /root/.cache/huggingface/hub
  Please run Cell 3 (Download HuggingFace Models) first
